In [ ]:
import random
from collections import defaultdict, Counter

D = """
the cat sat on the mat
the cat sat on the rug
the dog sat on the mat
the dog ran to the park
the cat ran to the park
the dog sat on the rug
"""
tokens = ["", ""] + list(set(D.split()))
print(f"Vocabulary: {tokens}\n")

# 1. Take a list of tokenized sentences as training data
# We split by line, then by word, and wrap each sentence with  and
raw_sentences = [line.strip().split() for line in D.strip().split('\n') if line.strip()]
training_data = [[""] + sentence + [""] for sentence in raw_sentences]

# 2. Count transitions between consecutive tokens
# We'll use a nested dictionary where transitions[current][next] = count
transitions = defaultdict(Counter)

for sentence in training_data:
    # Iterate through pairs of consecutive tokens
    for i in range(len(sentence) - 1):
        current_token = sentence[i]
        next_token = sentence[i + 1]
        transitions[current_token][next_token] += 1

# 3. Construct the conditional distribution P(Xt | Xt-1)
probabilities = defaultdict(dict)

for current_token, next_tokens in transitions.items():
    total_transitions = sum(next_tokens.values())
    for next_token, count in next_tokens.items():
        # P(Next | Current) = Count(Current -> Next) / Total transitions from Current
        probabilities[current_token][next_token] = count / total_transitions

# 4. Display the probabilities for a specified previous token
def show_probabilities(prev_token):
    print(f"P(X_t | X_t-1 = '{prev_token}'):")
    if prev_token not in probabilities:
        print("  Token not found in history.")
        return

    for next_token, prob in probabilities[prev_token].items():
        print(f"  {next_token}: {prob:.4f}")

# 5. Predict the most probable next token (Greedy / Argmax)
def predict_most_probable(prev_token):
    if prev_token not in probabilities:
        return None
    # Find the token with the highest probability value
    best_token = max(probabilities[prev_token].items(), key=lambda x: x[1])[0]
    return best_token

# 6 & 7. Generate a sentence by repeatedly sampling, stopping at
def generate_sentence():
    current_token = ""
    generated_tokens = []

    while True:
        # Get the possible next tokens and their corresponding probabilities
        next_choices = list(probabilities[current_token].keys())
        weights = list(probabilities[current_token].values())

        # Sample the next token based on the calculated distribution
        next_token = random.choices(next_choices, weights=weights, k=1)[0]

        # 7. Stop when the  token is generated
        if next_token == "":
            break

        generated_tokens.append(next_token)
        current_token = next_token

    return " ".join(generated_tokens)

# --- Demonstrating the functionality ---

# Show distribution for a specific token
show_probabilities("the")
print()

show_probabilities("cat")
print()

# Predict highest probability next token
prev = "sat"
best_next = predict_most_probable(prev)
print(f"Most probable token after '{prev}': '{best_next}'\n")

# Generate a few random sentences
print("Generated sentences:")
for i in range(5):
    print(f"{i+1}. {generate_sentence()}")

Vocabulary: ['', '', 'the', 'cat', 'dog', 'mat', 'sat', 'rug', 'park', 'ran', 'on', 'to']

P(X_t | X_t-1 = 'the'):
  cat: 0.2500
  mat: 0.1667
  rug: 0.1667
  dog: 0.2500
  park: 0.1667

P(X_t | X_t-1 = 'cat'):
  sat: 0.6667
  ran: 0.3333

Most probable token after 'sat': 'on'

Generated sentences:
1. the cat sat on the park
2. the dog sat on the dog sat on the rug
3. the cat sat on the cat sat on the dog sat on the rug
4. the dog ran to the park
5. the park
